In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False  # 让图表里的负号正常显示

# ===================== 关键修改：直接读取单个CSV =====================
file_path = r"../数据/沪深300成分股_10年.csv"  # 你刚刚导出的文件路径
df_all = pd.read_csv(file_path)

# 转换日期
df_all['trade_date'] = pd.to_datetime(df_all['trade_date'])

# 按股票代码 + 日期排序
df_all = df_all.sort_values(['ts_code', 'trade_date']).reset_index(drop=True)

import statsmodels.api as sm#线性回归工具

def neutralize_return_daily(df):
    """
    每天截面中性化：
    对数收益率 log_return 剔除 行业 + 市值 的影响
    输出：纯净收益率 log_return_neutral
    """
    def _neutralize(group):
        # 1. 市值取 log
        log_mv = np.log(group['market_cap'])

        # 2. 行业转哑变量，每一个行业赋值，便于识别
        ind_dummies = pd.get_dummies(group['industry'], drop_first=True)

        # 3. 构造回归 X
        X = pd.concat([ind_dummies, log_mv.rename('log_mv')], axis=1)
        X = sm.add_constant(X)

        # 4. 回归原始收益率
        y = group['log_return']

        try:
            # 残差 = 纯净收益率
            res = sm.OLS(y, X).fit().resid
        except:
            res = y  # 回归失败就用原值

        group['log_return_neutral'] = res
        return group

    # 按【每天】做截面中性化（最标准）
    return df.groupby('trade_date', group_keys=False).apply(_neutralize)

# 先计算原始 log_return
df_all['log_return'] = np.log(df_all['close'] / df_all['close'].shift(1))

# 执行中性化
df_all = neutralize_return_daily(df_all)

In [2]:

# 按股票分组，存入字典
all_stocks = {}
for code, group in df_all.groupby('ts_code'):
    df = group.copy().reset_index(drop=True)
    
    # 计算对数收益率
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    
    # 涨跌幅（直接用表中的 pct_chg）
    df['pct_return'] = df['pct_chg']
    
    all_stocks[code] = df
    print(f"加载 {code}: {len(df)} 条记录 | 日期范围 {df['trade_date'].min().date()} ~ {df['trade_date'].max().date()}")

print("\n✅ 全部股票加载完成！总股票数：", len(all_stocks))

加载 000001.SZ: 2427 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000002.SZ: 2336 条记录 | 日期范围 2016-07-04 ~ 2026-02-27
加载 000063.SZ: 2356 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000100.SZ: 2259 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000157.SZ: 2426 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000166.SZ: 2401 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000301.SZ: 2253 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000333.SZ: 2377 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000338.SZ: 2425 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000408.SZ: 2386 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000425.SZ: 2392 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000538.SZ: 2263 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000568.SZ: 2409 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000596.SZ: 2427 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000617.SZ: 2316 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000625.SZ: 2407 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000630.SZ: 2408 条记录 | 日期范围 2016-03-01 ~ 2026-02-27
加载 000651.SZ: 2276 条记录 | 日期范围 2016-09-02 ~ 2026-02-27
加载 000661.SZ: 2412 条记录 | 日期范

In [3]:
#量化分析每只股票的噪声特征
def analyze_noise_characteristics(stocks_dict):
    results = []
    
    for code, df in stocks_dict.items():
        returns = df['log_return_neutral'].dropna()
        
        # 1. 基础统计
        std = returns.std()# 波动率 = 噪声强度
        skew = returns.skew()# 偏度 = 涨跌不对称性
        kurt = returns.kurtosis()# 峰度 = 极端值多少
        
        # 2. 自相关性（衡量噪声的持续性）
        from scipy.stats import pearsonr
        autocorr_1 = returns.autocorr(lag=1) # 今日与昨日相关性
        autocorr_5 = returns.autocorr(lag=5)# 今日与5日前相关性
        #接近 0 = 市场有效、随机游走、纯噪声
        #不等于 0 = 有趋势、有规律
        # 3. 波动率聚集特征（用GARCH类型效应，简化为滚动标准差变异系数）
        rolling_std = returns.rolling(window=20).std().dropna()
        vol_clustering = rolling_std.std() / rolling_std.mean()  # CV越大，波动聚集越明显
        
        # 4. 极端值频率（超过3倍标准差的交易日占比）
        extreme_ratio = (abs(returns) > 3 * std).sum() / len(returns)
        
        # 5. 信息比率的倒数（噪声/信号）
        noise_to_signal = returns.std() / abs(returns.mean()) if returns.mean() != 0 else np.inf
        
        results.append({
            'code': code,
            '收益率标准差（波动率）': std,
            '偏度': skew,
            '峰度': kurt,
            '一阶自相关系数': autocorr_1,
            '五阶自相关系数': autocorr_5,
            '波动率聚集变异系数': vol_clustering,
            '极端值比例': extreme_ratio,
            '噪声信号比': noise_to_signal
        })
    
    return pd.DataFrame(results)

# 执行分析
noise_analysis = analyze_noise_characteristics(all_stocks)
noise_analysis = noise_analysis.sort_values('收益率标准差（波动率）', ascending=False)

print("\n=== 噪声特征分析结果 ===")
print(noise_analysis.to_string(index=False))

# 保存结果
noise_analysis.to_csv('../数据/噪声特征.csv', index=False, encoding='utf-8-sig')



=== 噪声特征分析结果 ===
     code  收益率标准差（波动率）         偏度          峰度   一阶自相关系数   五阶自相关系数  波动率聚集变异系数    极端值比例       噪声信号比
688256.SH     0.108846  28.037147  928.267001  0.001087  0.006353   0.704416 0.001523   28.292492
600522.SH     0.097588 -41.675543 1921.391796 -0.029507  0.005646   1.056130 0.000827   60.314735
688126.SH     0.096747 -30.538870 1067.504167 -0.042972 -0.242171   0.924229 0.002180   50.629059
603993.SH     0.096052 -42.773695 2004.706994 -0.034636  0.015739   0.847417 0.000416   91.484115
300316.SZ     0.091850 -39.936163 1830.945311 -0.030272 -0.024503   0.749028 0.001250   97.552593
300759.SZ     0.089490 -30.806983 1146.829572 -0.043193 -0.039944   0.677656 0.002365   61.840430
688169.SH     0.089074  28.027382  967.355939  0.016224  0.030394   0.652050 0.003482   67.848541
601136.SH     0.086690 -23.082974  602.230806 -0.084713 -0.113150   0.813636 0.001304   37.588942
600519.SH     0.081956  45.715911 2196.809450  0.026280 -0.019532   1.091945 0.000412   34.097732
00